In [5]:
# C1 - Imports et chargement du dataset

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("df_final.csv", parse_dates=["Date"])

print(f"Dimensions : {df.shape}")
print(f"Période    : {df['Date'].min().date()} à {df['Date'].max().date()}")
print(f"Colonnes   : {df.columns.tolist()}")

Dimensions : (408436, 30)
Période    : 2010-03-05 à 2012-10-26
Colonnes   : ['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday', 'Size', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'Week', 'Month', 'Year', 'Lag_1', 'Lag_2', 'Lag_4', 'Rolling_Mean_4', 'Rolling_Std_4', 'Black_Friday', 'Thanksgiving', 'Xmas_Week', 'Is_Promo', 'Type_A', 'Type_B', 'Type_C']


In [6]:
# C2 - Suppression des Weekly_Sales négatifs et transformation log

avant = len(df)
df = df[df["Weekly_Sales"] > 0].reset_index(drop=True)
apres = len(df)

print(f"Lignes avant suppression : {avant:,}")
print(f"Lignes supprimées        : {avant-apres:,} soit {(avant-apres)/avant*100:.2f}%")
print(f"Lignes conservées        : {apres:,} soit {apres/avant*100:.2f}%")
print(f"\nWeekly_Sales min après suppression : {df['Weekly_Sales'].min():.2f}$")

# Transformation log
df["Weekly_Sales_log"] = np.log(df["Weekly_Sales"])

print(f"\nWeekly_Sales_log — moyenne : {df['Weekly_Sales_log'].mean():.4f}")
print(f"Weekly_Sales_log — std     : {df['Weekly_Sales_log'].std():.4f}")

Lignes avant suppression : 408,436
Lignes supprimées        : 1,153 soit 0.28%
Lignes conservées        : 407,283 soit 99.72%

Weekly_Sales min après suppression : 0.01$

Weekly_Sales_log — moyenne : 8.5355
Weekly_Sales_log — std     : 2.0291



- 1 153 lignes supprimées soit 0.28% — négatifs ET zéros
- Justification scientifique : log(0) est mathématiquement indéfini
  La suppression des zéros est une contrainte de la transformation log pas un choix arbitraire
- 407 283 lignes conservées soit 99.72%
- np.log(Weekly_Sales) appliqué comme cible — corrige l'asymétrie de la distribution
- Weekly_Sales_log - moyenne 8.54, std 2.03
- Les prédictions seront reconverties avec np.exp() avant le calcul des métriques

In [7]:
# C3 - Fonction WMAE personnalisée

def wmae(y_true, y_pred, thanksgiving, black_friday, xmas_week):
    weights = pd.Series(1.0, index=y_true.index)
    weights[(thanksgiving == 1) | (black_friday == 1) | (xmas_week == 1)] = 5
    return round((weights * (y_true - y_pred).abs()).sum() / weights.sum(), 2)

- Pondération x5 : Thanksgiving, Black_Friday, Xmas_Week
- IsHoliday retiré - capte des fêtes sans impact réel sur les ventes
- Métriques calculées sur les valeurs réelles en dollars après np.exp()

In [8]:
# C4 - Définition des features, cible et validation

df = df.sort_values("Date").reset_index(drop=True)

features = [col for col in df.columns if col not in ["Date", "Weekly_Sales", "Weekly_Sales_log"]]
target   = "Weekly_Sales_log"

X            = df[features]
y            = df[target]
y_real       = df["Weekly_Sales"]
thanksgiving = df["Thanksgiving"]
black_friday = df["Black_Friday"]
xmas_week    = df["Xmas_Week"]

tscv = TimeSeriesSplit(n_splits=5)

print(f"Nombre de features : {len(features)}")
print(f"Lignes             : {len(X):,}")
print(f"Cible              : {target}")
print(f"Features           : {features}")

Nombre de features : 28
Lignes             : 407,283
Cible              : Weekly_Sales_log
Features           : ['Store', 'Dept', 'IsHoliday', 'Size', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'Week', 'Month', 'Year', 'Lag_1', 'Lag_2', 'Lag_4', 'Rolling_Mean_4', 'Rolling_Std_4', 'Black_Friday', 'Thanksgiving', 'Xmas_Week', 'Is_Promo', 'Type_A', 'Type_B', 'Type_C']


- 28 features - identiques à V1
- Cible : Weekly_Sales_log - valeurs en log
- y_real conservé pour reconversion np.exp() dans les métriques
- TimeSeriesSplit - 5 folds - ordre chronologique respecté

### C5 - Target Encoding - fonction réutilisable
- Moyennes de ventes calculées par Store, Dept et Store+Dept sur le fold train uniquement
- Appliquées ensuite sur la validation - les moyennes viennent du passé, pas du futur
- Pas de data leakage : la validation ne contribue jamais au calcul des moyennes
- Sécurité : combinaisons absentes du train remplacées par la moyenne globale

In [ ]:
# C5 - Target Encoding — fonction réutilisable

def target_encoding(X_train, X_val, y_train):
    
    # Clé Store+Dept
    X_train["Store_Dept_key"] = X_train["Store"].astype(str) + "_" + X_train["Dept"].astype(str)
    X_val["Store_Dept_key"]   = X_val["Store"].astype(str)   + "_" + X_val["Dept"].astype(str)

    # Moyennes sur le train uniquement
    store_mean      = y_train.groupby(X_train["Store"].values).mean()
    dept_mean       = y_train.groupby(X_train["Dept"].values).mean()
    store_dept_mean = y_train.groupby(X_train["Store_Dept_key"].values).mean()

    # Ajout au train
    X_train["Store_enc"]      = X_train["Store"].map(store_mean)
    X_train["Dept_enc"]       = X_train["Dept"].map(dept_mean)
    X_train["Store_Dept_enc"] = X_train["Store_Dept_key"].map(store_dept_mean)

    # Ajout à la validation
    X_val["Store_enc"]      = X_val["Store"].map(store_mean)
    X_val["Dept_enc"]       = X_val["Dept"].map(dept_mean)
    X_val["Store_Dept_enc"] = X_val["Store_Dept_key"].map(store_dept_mean)

    # Sécurité - combinaisons absentes du train
    global_mean = y_train.mean()
    X_val["Store_enc"]      = X_val["Store_enc"].fillna(global_mean)
    X_val["Dept_enc"]       = X_val["Dept_enc"].fillna(global_mean)
    X_val["Store_Dept_enc"] = X_val["Store_Dept_enc"].fillna(global_mean)

    # Suppression clé temporaire
    X_train = X_train.drop(columns=["Store_Dept_key"])
    X_val   = X_val.drop(columns=["Store_Dept_key"])

    return X_train, X_val

### C6 - Boucle CV - Random Forest V2 : suppression négatifs + log
- Cible : Weekly_Sales_log - modèle entraîné sur les valeurs en log
- Prédictions reconverties avec np.exp() avant calcul des métriques
- Toutes les métriques calculées en dollars réels - comparaison équitable avec V1
- Target Encoding appliqué à chaque fold
- Pondération WMAE : Thanksgiving, Black_Friday, Xmas_Week (x5)

In [10]:
# C6 - Boucle CV - Random Forest V2 : suppression négatifs + log

resultats_v2 = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X), 1):

    X_train  = X.iloc[train_idx].copy()
    X_val    = X.iloc[val_idx].copy()
    y_train  = y.iloc[train_idx]
    y_val    = y.iloc[val_idx]
    yr_val   = y_real.iloc[val_idx]

    # Variables WMAE
    t_val  = thanksgiving.iloc[val_idx]
    bf_val = black_friday.iloc[val_idx]
    xw_val = xmas_week.iloc[val_idx]

    # Target Encoding
    X_train, X_val = target_encoding(X_train, X_val, y_train)

    # Modèle - cible en log
    model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)

    # Prédiction en log puis reconversion en dollars réels
    y_pred_log  = model.predict(X_val)
    y_pred_real = np.exp(y_pred_log)
    y_pred_s    = pd.Series(y_pred_real, index=yr_val.index)

    # Métriques sur valeurs réelles
    resultats_v2.append({
        "Variante" : "V2 — Log + sans négatifs + Target Encoding",
        "Fold"     : f"Fold {fold}",
        "WMAE"     : wmae(yr_val, y_pred_s, t_val, bf_val, xw_val),
        "MAE"      : round(mean_absolute_error(yr_val, y_pred_real), 2),
        "RMSE"     : round(np.sqrt(mean_squared_error(yr_val, y_pred_real)), 2),
        "R2"       : round(r2_score(yr_val, y_pred_real), 4),
    })

    print(f"Fold {fold} — WMAE: {resultats_v2[-1]['WMAE']:,.0f}$ "
          f"MAE: {resultats_v2[-1]['MAE']:,.0f}$ "
          f"RMSE: {resultats_v2[-1]['RMSE']:,.0f}$ "
          f"R²: {resultats_v2[-1]['R2']:.4f}")

Fold 1 — WMAE: 4,313$ MAE: 2,843$ RMSE: 9,880$ R²: 0.8333
Fold 2 — WMAE: 1,491$ MAE: 1,491$ RMSE: 3,244$ R²: 0.9774
Fold 3 — WMAE: 2,445$ MAE: 1,703$ RMSE: 6,554$ R²: 0.9215
Fold 4 — WMAE: 1,845$ MAE: 1,663$ RMSE: 3,660$ R²: 0.9753
Fold 5 — WMAE: 1,318$ MAE: 1,318$ RMSE: 2,778$ R²: 0.9841


In [11]:
# C7 - Résultats et export V2

df_v2_results = pd.DataFrame(resultats_v2)

# Moyennes
print("--- Moyennes V2 ---")
print(df_v2_results[["WMAE","MAE","RMSE","R2"]].mean().round(2))

# Export
df_v2_results.to_csv("results_v2.csv", index=False)
print("\nExport results_v2.csv réussi")

--- Moyennes V2 ---
WMAE    2282.22
MAE     1803.58
RMSE    5223.44
R2         0.94
dtype: float64

Export results_v2.csv réussi
